In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d tapakah68/anti-spoofing

Dataset URL: https://www.kaggle.com/datasets/tapakah68/anti-spoofing
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 1.34G/1.34G [00:38<00:00, 37.5MB/s]



In [ ]:
!kaggle datasets download -d axondata/anti-spoofing-live-dataset

Dataset URL: https://www.kaggle.com/datasets/axondata/anti-spoofing-live-dataset
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 8.17M/8.17M [00:00<00:00, 10.6MB/s]



In [ ]:
!kaggle datasets download -d trainingdatapro/real-vs-fake-anti-spoofing-video-classification

Dataset URL: https://www.kaggle.com/datasets/trainingdatapro/real-vs-fake-anti-spoofing-video-classification
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 3.04G/3.04G [01:24<00:00, 38.6MB/s]



In [ ]:
!kaggle datasets download -d simongraves/deepfake-dataset

Dataset URL: https://www.kaggle.com/datasets/simongraves/deepfake-dataset
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 64.6M/64.6M [00:03<00:00, 20.3MB/s]



In [ ]:
import zipfile
zip = zipfile.ZipFile('/content/anti-spoofing.zip', 'r')
zip.extractall("/content/anti-spoofing/")
zip.close()


In [ ]:
import zipfile
zip = zipfile.ZipFile('/content/anti-spoofing-live-dataset.zip', 'r')
zip.extractall("/content/anti-spoofing/")
zip.close()


In [ ]:
import zipfile
zip = zipfile.ZipFile('/content/real-vs-fake-anti-spoofing-video-classification.zip', 'r')
zip.extractall("/content/real-vs-fake/")
zip.close()

In [1]:
import zipfile
zip = zipfile.ZipFile('/content/processed_dataset.zip', 'r')
zip.extractall("/content/processed_dataset/")
zip.close()

In [ ]:
shutil.rmtree('/content/deepfake')

In [ ]:
pd.read_csv('/content/real-vs-fake/real_and_fake.csv')

,file,type,split
0,train/real_video/0.mp4,real,train
1,train/attack/0.mp4,attack,train
2,train/real_video/1.mp4,real,train
3,train/attack/1.mp4,attack,train
4,train/real_video/2.mp4,real,train
...,...,...,...
155,test/attack/77.mp4,attack,test
156,test/real_video/78.mp4,real,test
157,test/attack/78.mp4,attack,test
158,test/real_video/79.mp4,real,test


In [1]:
import os
import numpy as np
import pandas as pd
import shutil
import cv2
import math
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers


In [ ]:


# 1. Define base paths
source_base = '/content/deepfake'
dest_base = '/content/anti-spoofing'

# 2. Define target destination paths
dest_printouts = os.path.join(dest_base, 'deepfakes')
dest_live_selfie = os.path.join(dest_base, 'live_selfie')
dest_live_video = os.path.join(dest_base, 'live_video')

# Ensure target directories exist
os.makedirs(dest_printouts, exist_ok=True)
os.makedirs(dest_live_selfie, exist_ok=True)
os.makedirs(dest_live_video, exist_ok=True)

# 3. Map specific subfolders to destinations
deepfake_mappings = {
    os.path.join(source_base, 'deepfake'): dest_printouts,
    os.path.join(source_base, 'image'): dest_live_selfie,
    os.path.join(source_base, 'video'): dest_live_video
}

# 4. Execute the move operation
for src_folder, dest_folder in deepfake_mappings.items():
    if not os.path.exists(src_folder):
        print(f"Skipping missing folder: {src_folder}")
        continue

    print(f"Moving files from deepfake/'{os.path.basename(src_folder)}' to '{os.path.basename(dest_folder)}'...")
    files = os.listdir(src_folder)

    for file_name in files:
        src_file = os.path.join(src_folder, file_name)
        dest_file = os.path.join(dest_folder, file_name)

        # Add a unique prefix to avoid overwriting any existing files with identical names
        if os.path.exists(dest_file):
            dest_file = os.path.join(dest_folder, "df_" + file_name)

        shutil.move(src_file, dest_file)

print("\nDeepfake dataset successfully sorted into anti-spoofing structure!")


Moving files from deepfake/'deepfake' to 'deepfakes'...
Moving files from deepfake/'image' to 'live_selfie'...
Moving files from deepfake/'video' to 'live_video'...

Deepfake dataset successfully sorted into anti-spoofing structure!


In [ ]:
# Define your source and destination folders
source_folder = '/content/anti-spoofing/AxonLabs_Selfie dataset - Kaggle'
destination_folder = '/content/anti-spoofing/live_selfie'

# Ensure the destination folder exists
os.makedirs(destination_folder, exist_ok=True)

# List all files in the source folder
files = os.listdir(source_folder)

# Move all image files (filter by extensions if needed)
for file in files:
    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
        source_path = os.path.join(source_folder, file)
        destination_path = os.path.join(destination_folder, file)
        shutil.move(source_path, destination_path)

print("All photos have been moved successfully!")

All photos have been moved successfully!


In [ ]:

# --- Configuration & Hyperparameters ---
BASE_DIR = '/content/anti-spoofing'
OUTPUT_BASE = '/content/processed_dataset'
IMG_SIZE = 224

# Mapping original subfolders to binary classes (Deepfake added here)
FOLDER_MAPPING = {
    'live_selfie': 'Live',
    'live_video': 'Live',
    'cut-out printouts': 'Attack',
    'printouts': 'Attack',
    'replay': 'Attack',
    'deepfake': 'Attack'  # <--- Added deepfake folder mapping
}

# Load the face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def process_and_save_wide_face(frame, save_path, scale_factor=1.5):
    """
    Detects the largest face and crops a WIDE bounding box around it
    to preserve crucial spoofing clues like screen bezels, edges, and hands.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)

    if len(faces) > 0:
        # Select the largest face detected
        x, y, w, h = max(faces, key=lambda b: b[2] * b[3])

        # Calculate centers
        cx, cy = x + w // 2, y + h // 2

        # Expand width and height dynamically by scale_factor (e.g., 1.5x)
        new_w = int(w * scale_factor)
        new_h = int(h * scale_factor)

        # Calculate new top-left coordinates ensuring they stay within frame boundaries
        img_h, img_w, _ = frame.shape
        new_x = max(0, cx - new_w // 2)
        new_y = max(0, cy - new_h // 2)

        # Ensure bottom-right coordinates stay within frame boundaries
        end_x = min(img_w, new_x + new_w)
        end_y = min(img_h, new_y + new_h)

        # Crop the wider region
        face_crop = frame[new_y:end_y, new_x:end_x]

        # Resize safely to your exact required model target size
        if face_crop.size > 0:
            resized = cv2.resize(face_crop, (IMG_SIZE, IMG_SIZE))
            cv2.imwrite(save_path, resized)
            return True

    return False

# --- Video-Level Dataset Partitioning Logic ---
# To stop leakage, we scan all files per folder first and separate the groups.
splits = ['train', 'val', 'test']
for s in splits:
    for c in ['Live', 'Attack']:
        os.makedirs(os.path.join(OUTPUT_BASE, s, c), exist_ok=True)

# Loop through each folder to divide videos systematically
for orig_folder, target_class in FOLDER_MAPPING.items():
    orig_path = os.path.join(BASE_DIR, orig_folder)
    if not os.path.exists(orig_path):
        print(f"Skipping missing directory: {orig_path}")
        continue

    # Gather and sort all individual target files to maintain alignment
    all_files = sorted([f for f in os.listdir(orig_path) if not f.startswith('.')])
    total_files = len(all_files)

    if total_files == 0:
        continue

    # Splitting math: 6 Train (approx 70%), 1 Val (approx 15%), 2 Test (approx 15%)
    # This splits the videos as standalone elements
    train_idx = math.ceil(total_files * 0.70)
    val_idx = train_idx + math.ceil(total_files * 0.15)

    train_files = all_files[:train_idx]
    val_files = all_files[train_idx:val_idx]
    test_files = all_files[val_idx:]

    print(f"\n📂 Partitioning {orig_folder} -> Target Class: {target_class}")
    print(f"   ↳ Train: {len(train_files)} items | Val: {len(val_files)} items | Test: {len(test_files)} items")

    # Map files explicitly to their designated final splits
    file_split_assignment = {}
    for f in train_files: file_split_assignment[f] = 'train'
    for f in val_files: file_split_assignment[f] = 'val'
    for f in test_files: file_split_assignment[f] = 'test'

    # Process all files with interactive real-time percentage progress bars
    for filename in tqdm(all_files, desc=f"Processing {orig_folder}", unit="file"):
        file_path = os.path.join(orig_path, filename)
        assigned_split = file_split_assignment[filename]
        target_dir = os.path.join(OUTPUT_BASE, assigned_split, target_class)

        # 1. Handle Static Selfie Images
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img = cv2.imread(file_path)
            if img is not None:
                save_name = f"img_{orig_folder}_{filename}"
                process_and_save_wide_face(img, os.path.join(target_dir, save_name))

        # 2. Handle Continuous Video Sequences
        elif filename.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
            cap = cv2.VideoCapture(file_path)
            frame_count = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                # Dynamic stride: sampling 1 frame every 15 frames to stop bulk file bloating
                if frame_count % 15 == 0:
                    base_name = os.path.splitext(filename)[0]
                    save_name = f"vid_{orig_folder}_{base_name}_f{frame_count}.jpg"
                    process_and_save_wide_face(frame, os.path.join(target_dir, save_name))
                frame_count += 1

            cap.release()

print("\n🚀 All frames successfully partitioned with wide dynamic boundaries!")



📂 Partitioning live_selfie -> Target Class: Live
   ↳ Train: 28 items | Val: 6 items | Test: 5 items


Processing live_selfie: 100%|██████████| 39/39 [00:48<00:00,  1.24s/file]



📂 Partitioning live_video -> Target Class: Live
   ↳ Train: 66 items | Val: 15 items | Test: 13 items


Processing live_video: 100%|██████████| 94/94 [16:34<00:00, 10.58s/file]



📂 Partitioning cut-out printouts -> Target Class: Attack
   ↳ Train: 7 items | Val: 2 items | Test: 0 items


...


📂 Partitioning printouts -> Target Class: Attack
   ↳ Train: 7 items | Val: 2 items | Test: 0 items


Processing printouts: 100%|██████████| 9/9 [05:04<00:00, 33.78s/file]



📂 Partitioning replay -> Target Class: Attack
   ↳ Train: 63 items | Val: 14 items | Test: 12 items


Processing replay: 100%|██████████| 89/89 [20:32<00:00, 13.85s/file]

Skipping missing directory: /content/anti-spoofing/deepfake

🚀 All frames successfully partitioned with wide dynamic boundaries!


In [ ]:

# 1. Compress the folder into a zip file
# Replace 'my_folder' with the exact name or path of your Colab folder
shutil.make_archive('/content/processed_dataset', 'zip', 'processed_dataset')

# 2. Download the zip file to your local computer
files.download('processed_dataset.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "processed_dataset/train",
    image_size = (224, 224),
    batch_size = 32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "processed_dataset/val",
    image_size = (224, 224),
    batch_size = 32
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "processed_dataset/test",
    image_size = (224, 224),
    batch_size = 32
)

Found 2812 files belonging to 2 classes.
Found 691 files belonging to 2 classes.
Found 547 files belonging to 2 classes.


In [3]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.03, fill_mode='nearest'),
    layers.RandomZoom(height_factor=(-0.08, 0.02), fill_mode='nearest'),
    layers.RandomBrightness(factor=0.08),
    layers.RandomContrast(factor=0.08)
])

In [4]:
from tensorflow.keras.applications import MobileNetV2


base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

base_model.trainable = False

In [5]:
model = tf.keras.Sequential([
    data_augmentation,
    layers.Rescaling(scale=1/127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
    ])

model.build(input_shape=(None, 224, 224, 3))


In [6]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=10)

In [10]:
import tensorflow as tf
# from google.colab import files
#
# class DownloadModelCallback(tf.keras.callbacks.Callback):
#     def on_train_end(self, logs=None):
#         print("\nTraining finished! Downloading model...")
#         files.download('my_mobilenet_v2.keras')
#
# checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
#     filepath='my_mobilenet_v2.keras',
#     save_best_only=True,
#     monitor='val_loss'
# )

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    # callbacks=[checkpoint_cb, DownloadModelCallback()]
)
model.save('my_model.keras')

Epoch 1/10
21/88 ━━━━━━━━━━━━━━━━━━━━ 17s 257ms/step - accuracy: 0.8994 - loss: 0.2441

KeyboardInterrupt: 